In [0]:
# ---------------------------------------------------------------------------
# SCHEMA CROSSWALK — machine-readable documentation of every variable
# mapping used to harmonize BRFSS 2024 and NYTS 2025 into the common Gold
# schema. This table IS the reusable artifact: any future CDC dataset that
# needs to join this framework gets mapped by adding rows here, not by
# reading through notebook code.
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

crosswalk_rows = [
    # (source_system, source_variable, source_label, harmonized_variable,
    #  transformation_logic, missing_value_convention, verified_against_codebook)
    ("BRFSS", "SEQNO", "Sequence number", "person_id", "Direct pass-through", "N/A", True),
    ("NYTS", "ARTIFICIAL_ID", "Artificial unique identifier", "person_id", "Direct pass-through", "N/A", True),

    ("BRFSS", "_LLCPWT", "Final BRFSS landline+cell weight", "survey_weight", "Cast to double", "N/A (weight always present)", True),
    ("NYTS", "WT_ANALYSIS", "NYTS analysis weight", "survey_weight", "Cast to double", "N/A (weight always present)", True),

    ("BRFSS", "SEXVAR", "Respondent sex", "sex", "Direct pass-through -- same 1=Male/2=Female convention as NYTS, no mapping needed", "None documented", True),
    ("NYTS", "Q2", "What is your sex?", "sex", "Direct pass-through -- same 1=Male/2=Female convention as BRFSS", "N=not answered -> null", True),

    ("BRFSS", "_AGEG5YR", "Reported age in five-year age categories", "age_category_raw", "Direct pass-through, kept as source scale", "N/A", True),
    ("NYTS", "Q1", "How old are you?", "age_category_raw", "Direct pass-through, kept as source scale", "N=not answered -> null", True),
    ("BOTH", "age_scale", "N/A", "age_scale", "Documents that BRFSS uses 5-year buckets and NYTS uses single years -- deliberately NOT harmonized into one scale, since populations don't overlap and false precision would misrepresent the data", "N/A", True),

    ("BRFSS", "_RACE", "Calculated race/ethnicity (single categorical)", "race_ethnicity",
     "Direct value-label mapping (1=White,2=Black,3=AIAN,4=Asian,5=NHPI,6=Other,7=Multiracial,8=Hispanic)",
     "None documented in this pass", False),  # NOT independently verified against 2024 codebook -- flagged
    ("NYTS", "Q4A-Q4G", "Race/ethnicity, select-all-that-apply (7 binary flags: AIAN/Asian/Black/Hispanic/MENA/NHPI/White)",
     "race_ethnicity",
     "Collapsed 7 flags to single category: exactly 1 flag=that category, >1 flag=Multiracial, 0 flags or any N/Z=Unknown. "
     "This collapse IS the real harmonization work -- BRFSS and NYTS structure this dimension completely differently.",
     ".=not selected (valid, ->0); N/Z=true missing (->null)", True),

    ("BRFSS", "_TOTINDA", "Leisure-time physical activity calculated variable", "primary_outcome_flag",
     "2 (no activity) -> 1; 1 (had activity) -> 0 -- recoded so 1 always means 'risk condition present'", "9=refused -> null", True),
    ("NYTS", "CELCIGT", "Current e-cigarette use, derived (>=1 of past 30 days)", "primary_outcome_flag",
     "1 (yes) -> 1; 2 (no) -> 0 -- same '1=risk present' convention as BRFSS", "M=missing -> null", True),

    ("BRFSS", "_STATE", "State FIPS code", "geography", "Direct pass-through", "N/A", True),
    ("NYTS", "N/A", "NYTS public-use file has no state-level identifier", "geography", "Always null for NYTS rows -- documented limitation, not a parsing gap", "N/A", True),
]

crosswalk_schema = ["source_system", "source_variable", "source_label", "harmonized_variable",
                     "transformation_logic", "missing_value_convention", "verified_against_codebook"]

crosswalk_df = spark.createDataFrame(crosswalk_rows, schema=crosswalk_schema)
crosswalk_df = crosswalk_df.withColumn("crosswalk_id", F.monotonically_increasing_id())

crosswalk_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_schema_crosswalk_metadata")
print(f"Written {crosswalk_df.count()} crosswalk entries")
crosswalk_df.orderBy("harmonized_variable", "source_system").show(40, truncate=60)

Written 15 crosswalk entries
+-------------+---------------+------------------------------------------------------------+--------------------+------------------------------------------------------------+------------------------------------------------------+-------------------------+------------+
|source_system|source_variable|                                                source_label| harmonized_variable|                                        transformation_logic|                              missing_value_convention|verified_against_codebook|crosswalk_id|
+-------------+---------------+------------------------------------------------------------+--------------------+------------------------------------------------------------+------------------------------------------------------+-------------------------+------------+
|        BRFSS|       _AGEG5YR|                    Reported age in five-year age categories|    age_category_raw|                   Direct pass-through, kept as sou

In [0]:
# Flag anything not yet independently verified against source codebooks --
# this is the honest "what still needs checking" query for the paper
print("Entries needing codebook verification before publication:")
crosswalk_df.filter(F.col("verified_against_codebook") == False).show(truncate=False)

Entries needing codebook verification before publication:
+-------------+---------------+----------------------------------------------+-------------------+---------------------------------------------------------------------------------------------------+----------------------------+-------------------------+------------+
|source_system|source_variable|source_label                                  |harmonized_variable|transformation_logic                                                                               |missing_value_convention    |verified_against_codebook|crosswalk_id|
+-------------+---------------+----------------------------------------------+-------------------+---------------------------------------------------------------------------------------------------+----------------------------+-------------------------+------------+
|BRFSS        |_RACE          |Calculated race/ethnicity (single categorical)|race_ethnicity     |Direct value-label mapping (1=White,2=Black

In [0]:
# Update the crosswalk entry now that it's independently verified
crosswalk_df = spark.table("workspace.default.gold_schema_crosswalk_metadata")

crosswalk_df = crosswalk_df.withColumn(
    "missing_value_convention",
    F.when(F.col("harmonized_variable") == "race_ethnicity",
           F.when(F.col("source_system") == "BRFSS", F.lit("9=refused/DK -> null")).otherwise(F.col("missing_value_convention")))
     .otherwise(F.col("missing_value_convention"))
).withColumn(
    "verified_against_codebook",
    F.when((F.col("source_system") == "BRFSS") & (F.col("harmonized_variable") == "race_ethnicity"), True)
     .otherwise(F.col("verified_against_codebook"))
)

crosswalk_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_schema_crosswalk_metadata")

print("Entries still needing verification:")
crosswalk_df.filter(F.col("verified_against_codebook") == False).show()

Entries still needing verification:
+-------------+---------------+------------+-------------------+--------------------+------------------------+-------------------------+------------+
|source_system|source_variable|source_label|harmonized_variable|transformation_logic|missing_value_convention|verified_against_codebook|crosswalk_id|
+-------------+---------------+------------+-------------------+--------------------+------------------------+-------------------------+------------+
+-------------+---------------+------------+-------------------+--------------------+------------------------+-------------------------+------------+

